In [ ]:
import cv2
import math
import os
import re
import json

In [ ]:
from Utils.DsFiles import *
from Utils.CvzUtils import *
from Utils import PlotUtils as PU

In [ ]:
PU.init_matplotlib_for_chinese()

In [ ]:
PATH_DUMP = "D:\\log\\MAIN_FPIX3\\Images\\20260727\\PositionFix"

In [ ]:
dsFiles = DsFiles(PATH_DUMP, ".bmp")

In [ ]:
def parse_index_row_col(filename: str):
    """
    解析檔名格式為 '任意字串_index@row_col.ext' 的字串
    
    :param filename: 檔名或路徑 (例如: "Cell_100@14_2.bmp", "A_B_C_100@14_2.png")
    :return: (index, row, col) 的整數元組 (tuple)，若格式不符則傳回 None
    """
    # 提取檔名主體 (去除資料夾路徑與副檔名)
    base_name = os.path.splitext(os.path.basename(filename))[0]
    
    # 正則表達式：
    # .*_    : 匹配任意前綴字串直到最後一個底線 (貪婪匹配)
    # (\d+)  : 匹配 index (數字)
    # @      : 分隔符號
    # (\d+)  : 匹配 row (數字)
    # _      : 分隔符號
    # (\d+)  : 匹配 col (數字)
    match = re.fullmatch(r".*_(\d+)@(\d+)_(\d+)", base_name)
    
    if match:
        idx, row, col = map(int, match.groups())
        return idx, row, col
    
    return (-1,-1,-1)

In [ ]:
def load_line_border_boxes(filename: str, encoding = 'utf-8-sig'):
    if not filename.endswith("_lines.json"):
        stem, ext = os.path.splitext(filename)
        filename = stem + "_lines.json"

    # 開啟並讀取 JSON 檔案
    with open(filename, 'r', encoding=encoding) as f:
        data = json.load(f)

    # 只過濾出 key 包含 'lineBorderBox' 的項目
    line_border_boxes = {
        key: value for key, value in data.items() if key.startswith('lineBorderBox')
    }
    
    for key, data in line_border_boxes.items():
        # reshape(-1, 2) 代表自動計算行數，固定欄數為 2
        points = np.array(data).reshape(-1, 2)
        line_border_boxes[key] = points

    return line_border_boxes

# 建立 {(row,col), imgFile}

In [ ]:
imgsDict = {}
rows = 0
cols = 0
goldenImgFile = None

In [ ]:
for imgFile, ds in dsFiles.items():
    fname = ds['fname']
    index, row, col = parse_index_row_col(fname)
    if index < 0:
        goldenImgFile = imgFile
        continue
    # print(fname, row, col)
    # continue
    imgsDict[(row,col)] = imgFile
    rows = max(rows, row + 1)
    cols = max(cols, col + 1)

print(f"(Rows, Cols) = ({rows}, {cols})")

In [ ]:
if False:
    for row in range(rows):
        imgs = []
        for col in range(cols):
            imgFile = imgsDict[(row,col)]
            img = cvz_imread(imgFile, gray=True)
            imgs.append(img)
        PU.smart_imshow_imgs(imgs, [f"row={row}"], sharey=True)


In [ ]:
if goldenImgFile is not None:
    imgFile = goldenImgFile
    img = cvz_imread(imgFile, gray= True)
    PU.smart_imshow(img=img, title = os.path.split(imgFile)[-1])

In [ ]:
imgFile = imgsDict[(0,5)]
img = cvz_imread(imgFile, gray= True)
PU.smart_imshow(img=img, title = os.path.split(imgFile)[-1])

In [ ]:
lboxes = load_line_border_boxes(imgFile)

img = cvz_imread(imgFile, gray=True)
canvas = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

for key, points in lboxes.items():
    pts = np.int32(points)  # OpenCV 畫圖座標需要整數 int32
    # [pts] 傳入包含點陣列的 List，isClosed=True 代表閉合多邊形
    cv2.polylines(canvas, [pts], isClosed=True, color=(255, 0, 0), thickness=3)
    pass


PU.smart_imshow(canvas)